# Ablation: plotting optimization axes

Kermit's *optimization standard* records which optimizations were active during a
benchmark as **axes** on each report, under three prefixes:

- `ds_layout_*` — compile-time layout (e.g. `ds_layout_hasher`)
- `ds_config_*` — runtime config flag (e.g. `ds_config_singleton_pruning`)
- `ds_build_mode` — construction-time build mode

`kl.load` surfaces every such axis as a DataFrame column, so you can put it on
any visual channel. An *ablation* figure holds the data structure / query fixed
and varies one optimization axis to isolate its effect.

> **Data note:** this notebook needs benchmark runs that actually *vary* an
> optimization axis — e.g. `HashTrie` swept across hashers, or a config flag
> toggled on/off. If your `bench-runs/` only has one setting per structure,
> the cells below detect that and explain what to run.

In [ ]:
import kermit_lab as kl

PATHS = '../../../bench-runs/*.json'
CRITERION_ROOT = '../../../target/criterion'

kl.apply_style()
df = kl.load(PATHS, criterion_root=CRITERION_ROOT)
print(f'{len(df)} rows, {len(df.columns)} columns')

## 1. Discover the optimization axes present

`kl.discover_opt_columns` returns the `ds_*` / `algo_*` columns in the frame.
We keep the ones with ≥2 distinct values — those are the ones worth ablating.

In [ ]:
opt_cols = kl.discover_opt_columns(df)
ablatable = [c for c in opt_cols if df[c].nunique(dropna=True) >= 2]
print('optimization axes present:', opt_cols)
print('ablatable (≥2 distinct values):', ablatable)

if not ablatable:
    print(
        '\nNothing to ablate here. Each --ds-layout-hasher run pins ONE hasher, '
        'so produce two reports and load both:\n'
        '  cargo run -- bench --report-json bench-runs/tri-sip.json \\\n'
        '      run triangle -i hash-trie -a hash-triejoin --ds-layout-hasher sip --metrics iteration\n'
        '  cargo run -- bench --report-json bench-runs/tri-fx.json \\\n'
        '      run triangle -i hash-trie -a hash-triejoin --ds-layout-hasher fxhash --metrics iteration\n'
        'then reload (ds_layout_hasher now has 2 distinct values).'
    )

## 2. The `ablation` preset

`kl.ablation(df, axis=...)` draws time vs the chosen optimization axis, coloured
by data structure, and facets by query when more than one is present.

In [ ]:
if ablatable:
    axis = ablatable[0]
    print(f'ablating on: {axis}')
    fig = kl.ablation(df, axis=axis)
    fig
else:
    axis = None
    print('skipped — see the note above')

## 3. The general engine

The preset is a thin configuration of `kl.plot`. Drive it directly to bind the
optimization axis to any channel — here `x` is the axis, colour is the data
structure, and we facet by query:

In [ ]:
if axis is not None:
    fig = kl.plot(
        df, kind='bar',
        x=axis, y='time',
        colour='data_structure',
        facet='query' if df['query'].nunique(dropna=True) > 1 else None,
    )
    fig
else:
    print('skipped — no ablatable axis')

## See also

- `02_scaling.ipynb` — time vs input size across data structures.
- `03_compare_ds.ipynb` — pairwise DS comparison with bootstrap CIs.
- `kl.plot(...)` is the general entry point; `kl.scaling`/`kl.bar_time`/
  `kl.ablation`/etc. are presets over it.